# VNTS HLS Training — Model B & Model P2
Second notebook in the series. Model A (winning HLS config) was trained and evaluated in the previous notebook. This one trains the two comparison models:

- **Model B** — plain YOLO26n baseline, no HLS (`cls=0.5`, flat one-hot targets). This is the same `B_baseline` config that was already in the CV ablation, just retrained on the full split for a final comparison.
- **Model P2** — YOLO26n with an added P2 detection head (better small-object resolution), also no HLS, so the only variable versus Model B is the architecture.

Both models train on **the exact same split Model A used** — attach the previous notebook's output as a Kaggle input dataset and point `SPLITS_INPUT` below at it. This isn't just "the same seed" — it's literally the same `data.yaml` / path-list files, so there's no risk of the split silently drifting between notebooks due to library-version differences.

Each model is evaluated on the 639-image test set exactly once, after training, exactly like Model A. The final cell pulls in Model A's exported summary and prints a three-way comparison table.

In [1]:
!pip install ultralytics wandb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.4 MB/s eta 0:00:00


## W&B logging setup
Identical to the previous notebook: logs in, defines `wandb_epoch_end`, attached via `on_fit_epoch_end` so it fires once both training losses and that epoch's validation metrics are available.

In [2]:
import wandb
from kaggle_secrets import UserSecretsClient
wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)


def wandb_epoch_end(trainer):
    """Log one epoch of Ultralytics metrics to W&B."""

    log = {"epoch": trainer.epoch + 1}

    if hasattr(trainer, "tloss"):
        names = ["train/box_loss", "train/cls_loss", "train/dfl_loss"]
        for i, v in enumerate(trainer.tloss):
            if i < len(names):
                log[names[i]] = float(v)

    if hasattr(trainer, "metrics"):
        try:
            for k, v in trainer.metrics.items():
                log[k] = float(v)
        except Exception:
            pass

    if hasattr(trainer, "lr"):
        for k, v in trainer.lr.items():
            log[f"lr/{k}"] = float(v)

    wandb.log(log)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vantu12772 (vantu12772-fpt-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Paths, constants, and the shared split
`SPLITS_INPUT` should point at the `exports/splits/` folder from the previous notebook's output (attach it as a Kaggle input dataset first — **update the path below to match the dataset slug Kaggle assigns it**). Everything under it — `train_val_data.yaml`, `test_data.yaml`, and the raw path-list `.txt` files — is copied verbatim into this notebook's working directory rather than recomputed, so Models B and P2 train/validate/test on identical data to Model A.

In [3]:
from pathlib import Path
import shutil
import numpy as np
import yaml
import torch
from ultralytics import YOLO

DATASET = Path("/kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive")
IMAGES  = DATASET / "images"
LABELS  = DATASET / "labels"
CLASSES = DATASET / "classes_en.txt"

# UPDATE THIS to match the Kaggle dataset slug for notebook 1's output.
SPLITS_INPUT = Path("/kaggle/input/datasets/banutheghostbanana/model-a-hls/exports/splits")

WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True, parents=True)

with open(CLASSES, encoding="utf-8") as f:
    class_names = [l.strip() for l in f if l.strip()]
n_classes = len(class_names)

SEED = 0

# Copy the shared split locally so relative "path:" entries inside the
# yaml files resolve correctly regardless of where SPLITS_INPUT lives.
SPLIT_WORK = WORK / "split"
SPLIT_WORK.mkdir(exist_ok=True, parents=True)
for fname in ["train_val_data.yaml", "train_paths.txt", "val_paths.txt",
              "test_data.yaml", "test_paths.txt"]:
    shutil.copy(SPLITS_INPUT / fname, SPLIT_WORK / fname)

# Patch the copied yaml files' "path" field to point at SPLIT_WORK, since
# they were originally written relative to notebook 1's working directory.
for yaml_name in ["train_val_data.yaml", "test_data.yaml"]:
    yaml_path = SPLIT_WORK / yaml_name
    with open(yaml_path, encoding="utf-8") as f:
        data = yaml.safe_load(f)
    data["path"] = str(SPLIT_WORK)
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(data, f, allow_unicode=True, sort_keys=False)

train_val_yaml_path = SPLIT_WORK / "train_val_data.yaml"
test_yaml_path = SPLIT_WORK / "test_data.yaml"

with open(SPLIT_WORK / "train_paths.txt", encoding="utf-8") as f:
    n_final_train = len(f.readlines())
with open(SPLIT_WORK / "val_paths.txt", encoding="utf-8") as f:
    n_internal_val = len(f.readlines())
with open(SPLIT_WORK / "test_paths.txt", encoding="utf-8") as f:
    n_test = len(f.readlines())

print(f"Classes: {n_classes}")
print(f"Final training pool: {n_final_train} | Internal val: {n_internal_val} | Test: {n_test}")
print(f"train_val_data.yaml -> {train_val_yaml_path}")
print(f"test_data.yaml      -> {test_yaml_path}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Classes: 52
Final training pool: 2305 | Internal val: 247 | Test: 639
train_val_data.yaml -> /kaggle/working/split/train_val_data.yaml
test_data.yaml      -> /kaggle/working/split/test_data.yaml


## Frequency tiers (Frequent / Medium / Rare)
Recomputed the same way as in notebook 1 — this is a deterministic count over the label files, no randomness involved, so it doesn't need to be imported from the shared split export.

In [4]:
with open(SPLIT_WORK / "train_paths.txt", encoding="utf-8") as f:
    all_paths = [l.strip() for l in f if l.strip()]
with open(SPLIT_WORK / "val_paths.txt", encoding="utf-8") as f:
    all_paths += [l.strip() for l in f if l.strip()]
with open(SPLIT_WORK / "test_paths.txt", encoding="utf-8") as f:
    all_paths += [l.strip() for l in f if l.strip()]

def count_instances(paths):
    counts = np.zeros(n_classes, dtype=int)
    for p in paths:
        label_path = LABELS / (Path(p).stem + ".txt")
        if label_path.exists():
            with open(label_path, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        counts[int(line.split()[0])] += 1
    return counts

all_counts = count_instances(all_paths)

FREQ_CLASS_IDS = np.where(all_counts >= 100)[0]
MED_CLASS_IDS  = np.where((all_counts >= 30) & (all_counts < 100))[0]
RARE_CLASS_IDS = np.where(all_counts < 30)[0]

def per_tier_map50(metrics_box, n_classes=n_classes):
    """AP50 (column 0 of all_ap) averaged per frequency tier.

    Ultralytics drops rows for classes with zero ground-truth instances in
    this val split rather than zero/NaN-filling them, so `all_ap` isn't
    reliably indexable by raw class ID. `ap_class_index` gives the true
    class ID per row; scatter back into a full-length NaN-filled array
    before slicing by tier (matches notebook 1's fix for the same bug).
    """
    full = np.full(n_classes, np.nan)
    ap50_present = metrics_box.all_ap[:, 0]
    class_idx = np.asarray(metrics_box.ap_class_index)
    full[class_idx] = ap50_present
    return {
        "global": float(np.nanmean(full)),
        "freq":   float(np.nanmean(full[FREQ_CLASS_IDS])),
        "med":    float(np.nanmean(full[MED_CLASS_IDS])),
        "rare":   float(np.nanmean(full[RARE_CLASS_IDS])),
    }

print(f"Frequent: {len(FREQ_CLASS_IDS)} classes | Medium: {len(MED_CLASS_IDS)} classes "
      f"| Rare: {len(RARE_CLASS_IDS)} classes")


Frequent: 24 classes | Medium: 15 classes | Rare: 13 classes


## FINAL_EPOCHS and shared training defaults
Same 100-epoch budget as Model A. If Model P2's per-epoch time comes in noticeably higher than Model A/B's (the extra P2 head adds compute), check wall-clock after Model P2's first few epochs before assuming the full run will fit your time budget.

In [5]:
FINAL_EPOCHS = 100
BASE_TRAIN_KWARGS = dict(
    epochs=FINAL_EPOCHS,
    batch=32,
    imgsz=640,
    lr0=0.01,
    lrf=0.01,
    seed=SEED,
    label_smoothing=0.0,
    exist_ok=True,
    verbose=True,
)  # cls defaults to 0.5 (ultralytics default) since both models here are baseline, no HLS


## Train Model B (plain YOLO26n baseline)
No HLS patch, no custom `cls` weight — this is the flat one-hot-target baseline. Same W&B project as Model A (`vnts-hls-final`) so all three final runs live side by side, distinguished by group.

In [6]:
model_b_run_name = "ModelB_baseline_final"

wandb.init(
    project="vnts-hls-final",
    group="ModelB",
    name=model_b_run_name,
    job_type="final_train",
    tags=["Final-Train", "YOLO26n", "ModelB", "Baseline"],
    reinit=True,
    config={
        "configuration": "B_baseline",
        "architecture": "YOLO26n",
        "pretrained_weights": "yolo26n.pt",
        "use_hls": False,
        "epochs": FINAL_EPOCHS,
        "batch_size": 32,
        "image_size": 640,
        "learning_rate": 0.01,
        "final_lr_factor": 0.01,
        "random_seed": SEED,
    }
)

model_b = YOLO("yolo26n.pt")
model_b.add_callback("on_fit_epoch_end", wandb_epoch_end)

model_b.train(
    data=str(train_val_yaml_path),
    project=str(WORK / "final_modelB" / "runs"),
    name=model_b_run_name,
    **BASE_TRAIN_KWARGS,
)

model_b_best = WORK / "final_modelB" / "runs" / model_b_run_name / "weights" / "best.pt"
print(f"Model B best checkpoint: {model_b_best}")


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260712_110331-8n6d3b50
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ModelB_baseline_final
wandb: ⭐️ View project at https://wandb.ai/vantu12772-fpt-university/vnts-hls-final
wandb: 🚀 View run at https://wandb.ai/vantu12772-fpt-university/vnts-hls-final/runs/8n6d3b50


WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/split/train_val_data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, 

## Evaluate Model B on the test set (exactly once)

In [7]:
model_b_final = YOLO(str(model_b_best))
b_test_metrics = model_b_final.val(data=str(test_yaml_path), split="val")
b_test_tier_scores = per_tier_map50(b_test_metrics.box)

print("=== Model B -- FINAL TEST SET RESULTS (one-time evaluation) ===")
for k, v in b_test_tier_scores.items():
    print(f"  {k:6s}: {v:.4f}")

wandb.log({
    "test/global_mAP50": b_test_tier_scores["global"],
    "test/freq_mAP50":   b_test_tier_scores["freq"],
    "test/med_mAP50":    b_test_tier_scores["med"],
    "test/rare_mAP50":   b_test_tier_scores["rare"],
})
wandb.finish()


Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,384,976 parameters, 0 gradients, 5.2 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 2.9±2.0 ms, read: 42.7±20.1 MB/s, size: 291.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive/labels... 639 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 639/639 278.9it/s 2.3s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 8.2it/s 4.9s
                   all        639       1645      0.834       0.88      0.926      0.695
   Pedestrian Crossing         61         61       0.88      0.934      0.957      0.

wandb: updating run metadata
wandb: 
wandb: Run history:
wandb:                epoch ▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▇▇▇▇▇███
wandb:            lr/lr/pg0 █████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
wandb:            lr/lr/pg1 ▃████▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▁▁▁▁
wandb:            lr/lr/pg2 ▃████▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
wandb:     metrics/mAP50(B) ▁▂▃▄▅▆▆▆▆▇▇▇▇▇▇█████████████████████████
wandb:  metrics/mAP50-95(B) ▁▂▃▃▄▅▅▅▅▆▆▆▇▇▇▇█▇██████████████████████
wandb: metrics/precision(B) ▁▇▇▆▆▆▅▇▆▆▇▇▇▇▇▇▇▇▇▇▇▇██▇██▇▇▇▇█▇█▇█▇███
wandb:    metrics/recall(B) ▁▁▃▄▅▆▆▆▇▇▇▇▇▇▇█▇▇▇██▇█▇████████▇██▇▇▇▇█
wandb:      test/freq_mAP50 ▁
wandb:    test/global_mAP50 ▁
wandb:                   +8 ...
wandb: 
wandb: Run summary:
wandb:                epoch 101
wandb:            lr/lr/pg0 0.0
wandb:            lr/lr/pg1 0.0
wandb:            lr/lr/pg2 0.0
wandb:     metrics/mAP50(B) 0.92834
wandb:  metrics/mAP50-95(B) 0.71416
wandb: metrics/precision(B) 0.79209
wandb:    metrics/recall(B) 0.86594

## Train Model P2 (YOLO26n with P2 head)
Built from the `yolo26n-p2.yaml` architecture config with weights transferred from `yolo26n.pt` where shapes match, leaving the new P2 layers randomly initialized — the standard Ultralytics pattern for architecture-modified transfer learning (`YOLO(cfg).load(weights)`).

**Please verify `yolo26n-p2.yaml` is the correct config filename for your installed Ultralytics version before running this cell** — I can't check your package's `cfg/models/` directory from here, and P2-head naming has varied across Ultralytics releases (e.g. `yolov8-p2.yaml` historically). Run `from ultralytics.utils import checks; checks.check_yolo()` or browse your installed package's `ultralytics/cfg/models/` folder if the load fails, and swap the filename below.

In [8]:
model_p2_run_name = "ModelP2_baseline_final"

wandb.init(
    project="vnts-hls-final",
    group="ModelP2",
    name=model_p2_run_name,
    job_type="final_train",
    tags=["Final-Train", "YOLO26n-P2", "ModelP2", "Baseline"],
    reinit=True,
    config={
        "configuration": "P2_baseline",
        "architecture": "YOLO26n-P2",
        "base_cfg": "yolo26n-p2.yaml",
        "pretrained_weights": "yolo26n.pt",
        "use_hls": False,
        "epochs": FINAL_EPOCHS,
        "batch_size": 32,
        "image_size": 640,
        "learning_rate": 0.01,
        "final_lr_factor": 0.01,
        "random_seed": SEED,
    }
)

model_p2 = YOLO("yolo26n-p2.yaml").load("yolo26n.pt")
model_p2.add_callback("on_fit_epoch_end", wandb_epoch_end)

model_p2.train(
    data=str(train_val_yaml_path),
    project=str(WORK / "final_modelP2" / "runs"),
    name=model_p2_run_name,
    **BASE_TRAIN_KWARGS,
)

model_p2_best = WORK / "final_modelP2" / "runs" / model_p2_run_name / "weights" / "best.pt"
print(f"Model P2 best checkpoint: {model_p2_best}")


wandb: setting up run vj2vljys
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260712_115824-vj2vljys
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ModelP2_baseline_final
wandb: ⭐️ View project at https://wandb.ai/vantu12772-fpt-university/vnts-hls-final
wandb: 🚀 View run at https://wandb.ai/vantu12772-fpt-university/vnts-hls-final/runs/vj2vljys


Transferred 360/902 items from pretrained weights
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/split/train_val_data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n

## Evaluate Model P2 on the test set (exactly once)

In [9]:
model_p2_final = YOLO(str(model_p2_best))
p2_test_metrics = model_p2_final.val(data=str(test_yaml_path), split="val")
p2_test_tier_scores = per_tier_map50(p2_test_metrics.box)

print("=== Model P2 -- FINAL TEST SET RESULTS (one-time evaluation) ===")
for k, v in p2_test_tier_scores.items():
    print(f"  {k:6s}: {v:.4f}")

wandb.log({
    "test/global_mAP50": p2_test_tier_scores["global"],
    "test/freq_mAP50":   p2_test_tier_scores["freq"],
    "test/med_mAP50":    p2_test_tier_scores["med"],
    "test/rare_mAP50":   p2_test_tier_scores["rare"],
})
wandb.finish()


Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n-p2 summary (fused): 152 layers, 2,429,832 parameters, 0 gradients, 6.8 GFLOPs
val: Fast image access ✅ (ping: 1.7±0.8 ms, read: 466.0±204.9 MB/s, size: 238.7 KB)
val: Scanning /kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive/labels... 639 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 639/639 625.3it/s 1.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 7.1it/s 5.6s
                   all        639       1645      0.845      0.671      0.833      0.617
   Pedestrian Crossing         61         61      0.846      0.852      0.935       0.67
Equal-level Intersection          5          5      0.605        0.4      0.558      0.355
              No Entry         87         88      0.823 

wandb: updating run metadata
wandb: 
wandb: Run history:
wandb:                epoch ▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
wandb:            lr/lr/pg0 ███▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▁▁▁▁
wandb:            lr/lr/pg1 ▆████▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁
wandb:            lr/lr/pg2 ▃████▇▇▇▇▇▇▆▆▅▅▅▅▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁
wandb:     metrics/mAP50(B) ▁▁▁▂▂▃▃▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇███████████████
wandb:  metrics/mAP50-95(B) ▁▁▁▁▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇▇████████████████████
wandb: metrics/precision(B) ▁▅▅▅▆▅▅▆▆▅▆▇▇▇▆▇▇▇▇███▇█▇██▇▇▇▇█▇▇█▇▆▆▇▇
wandb:    metrics/recall(B) ▁▂▂▂▂▃▄▄▄▄▄▄▅▅▆▅▅▅▆▆▆▆▆▆▆▇▆▇▆▇▇▇▇█▇█████
wandb:      test/freq_mAP50 ▁
wandb:    test/global_mAP50 ▁
wandb:                   +8 ...
wandb: 
wandb: Run summary:
wandb:                epoch 101
wandb:            lr/lr/pg0 0.0
wandb:            lr/lr/pg1 0.0
wandb:            lr/lr/pg2 0.0
wandb:     metrics/mAP50(B) 0.82237
wandb:  metrics/mAP50-95(B) 0.62023
wandb: metrics/precision(B) 0.79567
wandb:    metrics/recall(B) 0.7279


## Export Model B & P2 outputs

In [10]:
import json

for tag, best_ckpt, tier_scores, cfg_name in [
    ("model_B",  model_b_best,  b_test_tier_scores,  "B_baseline"),
    ("model_P2", model_p2_best, p2_test_tier_scores, "P2_baseline"),
]:
    export_dir = WORK / "exports" / tag
    export_dir.mkdir(exist_ok=True, parents=True)
    shutil.copy(best_ckpt, export_dir / f"{tag}_best.pt")

    summary = {
        "model": tag,
        "config_name": cfg_name,
        "hyperparameters": {"hls": False},
        "final_train_epochs": FINAL_EPOCHS,
        "final_test_results": tier_scores,
    }
    with open(export_dir / f"{tag}_summary.json", "w") as f:
        json.dump(summary, f, indent=4)

    print(f"{tag} outputs exported to: {export_dir}")


model_B outputs exported to: /kaggle/working/exports/model_B
model_P2 outputs exported to: /kaggle/working/exports/model_P2


## Three-way comparison: A vs B vs P2
Pulls in Model A's summary (attach the same notebook-1 output dataset used for the split, since `model_A_summary.json` lives alongside it in `exports/model_A/`) and lines up all three models' one-time test results side by side.

In [11]:
# UPDATE THIS to match wherever notebook 1's exports/model_A/ folder landed
MODEL_A_SUMMARY_PATH = Path("/kaggle/input/datasets/banutheghostbanana/model-a-hls/exports/model_A/model_A_summary.json")

with open(MODEL_A_SUMMARY_PATH) as f:
    model_a_summary = json.load(f)

rows = [
    ("A", model_a_summary["winning_config_name"], model_a_summary["final_test_results"]),
    ("B", "B_baseline", b_test_tier_scores),
    ("P2", "P2_baseline", p2_test_tier_scores),
]

print(f"{'Model':6s} {'Config':16s} {'Global':>8s} {'Freq':>8s} {'Med':>8s} {'Rare':>8s}")
for name, cfg, scores in rows:
    print(f"{name:6s} {cfg:16s} {scores['global']:8.4f} {scores['freq']:8.4f} "
          f"{scores['med']:8.4f} {scores['rare']:8.4f}")


Model  Config             Global     Freq      Med     Rare
A      HLS-3_hicls        0.9131   0.9556   0.9387   0.8053
B      B_baseline         0.9257   0.9536   0.9336   0.8649
P2     P2_baseline        0.8335   0.8839   0.8523   0.7188
